In [1]:
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
df = pd.read_csv("../dataset_metadata.csv")
print("Total samples:", len(df))
df.head()

Total samples: 1440


,path,emotion,actor
0,../dataset\Actor_01\03-01-01-01-01-01-01.wav,neutral,Actor_01
1,../dataset\Actor_01\03-01-01-01-01-02-01.wav,neutral,Actor_01
2,../dataset\Actor_01\03-01-01-01-02-01-01.wav,neutral,Actor_01
3,../dataset\Actor_01\03-01-01-01-02-02-01.wav,neutral,Actor_01
4,../dataset\Actor_01\03-01-02-01-01-01-01.wav,calm,Actor_01


In [3]:
def add_noise(audio):
    noise = np.random.normal(0,0.02, audio.shape)
    return audio + noise

def pitch_shift(audio, sr):
    return librosa.effects.pitch_shift(audio,sr=sr,n_steps=2)

def extract_logmel_from_audio(audio, sr):
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=128,
        hop_length=512
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)
    return log_mel

In [4]:
#Extraction of spectogram dataset + adding padding
X=[]
y=[]
max_len = 128
for index, row in tqdm(df.iterrows(), total=len(df)):
    audio, sr = librosa.load(row["path"], sr=22050)
    #Original + Augmented versions
    variants = [audio, add_noise(audio), pitch_shift(audio, sr)]
    for variant in variants:
        spec = extract_logmel_from_audio(variant, sr)
        #pad or truncate for inputs to have identical shape
        if spec.shape[1] < max_len:
            pad_width = max_len - spec.shape[1]
            spec = np.pad(spec, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            spec = spec[:, :max_len]
        X.append(spec)
        y.append(row["emotion"])

X = np.array(X)
y = np.array(y)

print("Shape:",X.shape)

  0%|                                                                         | 0/1440 [00:00<?, ?it/s]C:\ddata\git program\speech-emotion-recognition\ser_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████████████████████████████████████████████████████████| 1440/1440 [04:10<00:00,  5.75it/s]


Shape: (4320, 128, 128)


In [6]:
# Get emotion labels directly
emotion_labels = np.unique(y)
print("Classes:", emotion_labels)

Classes: ['angry' 'calm' 'disgust' 'fearful' 'happy' 'neutral' 'sad' 'surprised']


In [27]:
# Flatten (128x128 to 16384)
X_rf = X.reshape(X.shape[0], -1)
print("RF Input Shape:", X_rf.shape)
# PCA
from sklearn.decomposition import PCA
pca = PCA(n_components=200)
X_rf = pca.fit_transform(X_rf)

RF Input Shape: (4320, 16384)


In [28]:
#Train test split
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(X_rf, y,test_size=0.2,random_state=42,stratify=y)

In [29]:
#Training model
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42
)
print("Training Random Forest...")
rf_model.fit(X_train_rf, y_train_rf)

Training Random Forest...


,n_estimators,200
,criterion,'gini'
,max_depth,20
,min_samples_split,5
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [30]:
#model accuracy
y_pred_rf = rf_model.predict(X_test_rf)
rf_accuracy = accuracy_score(y_test_rf, y_pred_rf)
print("Random Forest Accuracy:", rf_accuracy)

Random Forest Accuracy: 0.6111111111111112


In [31]:
#classification report
rf_report_dict = classification_report(
    y_test_rf,
    y_pred_rf,
    labels=emotion_labels,
    output_dict=True
)
rf_report_df = pd.DataFrame(rf_report_dict).transpose()
rf_report_df["Model"] = "Random Forest"
rf_report_df.to_csv("rf_classification_report.csv")

In [32]:
cm = confusion_matrix(y_test_rf, y_pred_rf, labels=emotion_labels)
cm_df = pd.DataFrame(cm, index=emotion_labels, columns=emotion_labels)
cm_df.to_csv("rf_confusion_matrix.csv")

In [33]:
rf_summary = pd.DataFrame({
    "Model": ["Random Forest"],
    "Accuracy": [rf_accuracy]
})
print(rf_summary)
rf_summary.to_csv("rf_summary.csv", index=False)

           Model  Accuracy
0  Random Forest  0.611111
